In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!nvidia-smi

Tue Mar 24 21:22:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# Clone repo (or pull latest)
import os

REPO_DIR = '/content/aphasia-modeling'
DRIVE_DIR = '/content/drive/MyDrive/aphasia-modeling'

# Create drive dir for persistent data
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    # TODO: replace with your repo URL
    !git clone https://github.com/evandiewald/aphasia-modeling.git {REPO_DIR}

Already up to date.


In [4]:
# Install dependencies
!cd {REPO_DIR} && pip install -e '.[dev]' wandb -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for aphasia-modeling (pyproject.toml) ... done


In [ ]:
import os

os.environ['WANDB_API_KEY'] = 'REPLACE_ME'
os.environ['HF_TOKEN'] = 'REPLACE_ME'

In [13]:
# Option A: Copy data from Drive to local (faster I/O during training)
DATA_DIR = f'{REPO_DIR}/data'
os.makedirs(f'{DATA_DIR}/Fridriksson/audio', exist_ok=True)

!cp {DRIVE_DIR}/data/fridriksson.json {DATA_DIR}/fridriksson.json
!cp {DRIVE_DIR}/data/Fridriksson/audio/*.wav {DATA_DIR}/Fridriksson/audio/

# Verify
!ls -la {DATA_DIR}/fridriksson.json
!ls {DATA_DIR}/Fridriksson/audio/ | wc -l
print('audio files found ^')

-rw------- 1 root root 1462300 Mar 24 21:27 /content/aphasia-modeling/data/fridriksson.json
17
audio files found ^


In [14]:
# Update audio paths in the dataset to point to local copies
import json
from pathlib import Path

data_path = Path(f'{DATA_DIR}/fridriksson.json')
records = json.loads(data_path.read_text())

audio_dir = Path(f'{DATA_DIR}/Fridriksson/audio')
updated = 0
for r in records:
    if r.get('audio_path'):
        # Rewrite path to point to local audio dir
        wav_name = Path(r['audio_path']).name
        new_path = str(audio_dir / wav_name)
        if Path(new_path).exists():
            r['audio_path'] = new_path
            updated += 1

data_path.write_text(json.dumps(records, indent=2))
print(f'Updated {updated}/{len(records)} audio paths')

Updated 2808/2808 audio paths


In [19]:
!cd {REPO_DIR} && git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 572 bytes | 572.00 KiB/s, done.
From https://github.com/evandiewald/aphasia-modeling
   7ab8488..4b3897a  main       -> origin/main
Updating 7ab8488..4b3897a
Fast-forward
 src/aphasia_modeling/model/trainer.py | 4 +++-
 1 file changed, 3 insertions(+), 1 deletion(-)


In [ ]:
!cd {REPO_DIR} && python scripts/train.py \
    --phase 2 \
    --data_path data/fridriksson.json \
    --model_name openai/whisper-small \
    --test_speaker fridriksson01 \
    --output_dir {DRIVE_DIR}/checkpoints/fold_fridriksson01 \
    --class_weights \
    --bf16 \
    --num_workers 2 \
    --wandb

Loading dataset from data/fridriksson.json...
Loaded 2808 utterances, 13 speakers: ['fridriksson01', 'fridriksson02', 'fridriksson03', 'fridriksson04', 'fridriksson05', 'fridriksson06', 'fridriksson07', 'fridriksson08', 'fridriksson09', 'fridriksson10', 'fridriksson11', 'fridriksson12', 'fridriksson13']

--- Phase 2: Paraphasia Fine-tuning ---

Training fold: test_speaker=fridriksson01
Output: /content/drive/MyDrive/aphasia-modeling/checkpoints/fold_fridriksson01

Train: 2274 utterances
Dev:   252 utterances
Test:  282 utterances
Loading weights: 100% 479/479 [00:00<00:00, 866.84it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API